## 权重比较

In [5]:
import torch
import numpy as np
from safetensors.torch import load_file
from torch.serialization import add_safe_globals

# 允许 numpy scalar 类型（解决 UnpicklingError）
add_safe_globals([np.core.multiarray.scalar])

# 路径
ckpt_path = "/home/xiaoxinyu/nicheformer/data/checkpoint/nicheformer.ckpt"
safetensors_path = "/data2/xiaoxinyu/nicheformer-hg/model.safetensors"

# 读取 ckpt
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
ckpt_state = ckpt["state_dict"] if "state_dict" in ckpt else ckpt

# 读取 safetensors
safetensors_state = load_file(safetensors_path)

# 打印基本信息
print(f"ckpt 参数量: {len(ckpt_state)}")
print(f"safetensors 参数量: {len(safetensors_state)}")

# 比较 key
ckpt_keys = set(ckpt_state.keys())
safetensors_keys = set(safetensors_state.keys())

common_keys = ckpt_keys & safetensors_keys
print(f"共有 key 数: {len(common_keys)}")

# 检查参数是否一致
diff = []
for k in common_keys:
    if not torch.equal(ckpt_state[k].cpu(), safetensors_state[k].cpu()):
        diff.append(k)

if len(diff) == 0:
    print("✅ 两个文件的参数完全一致")
else:
    print(f"⚠️ 有 {len(diff)} 个参数不一致，例如: {diff[:5]}")


ckpt 参数量: 164
safetensors 参数量: 161
共有 key 数: 2
⚠️ 有 2 个参数不一致，例如: ['classifier_head.weight', 'classifier_head.bias']


## 基因名转换

In [8]:
import anndata as ad
import mygene
import pandas as pd

# 读取数据
adata = ad.read_h5ad("/data2/xiaoxinyu/project/embedding-cosine/single-spot/sc2.h5ad")
symbols = list(adata.var_names)

# 使用 mygene 查询
mg = mygene.MyGeneInfo()
res = mg.querymany(symbols, scopes='symbol', fields='ensembl.gene', species='mouse')

# 构造映射：symbol -> ensembl gene id
mapping = {}
for entry in res:
    sym = entry['query']
    ens = None
    if 'ensembl' in entry:
        e = entry['ensembl']
        ens = e['gene'] if isinstance(e, dict) else e[0]['gene']
    mapping[sym] = ens or sym  # 找不到就保留原 symbol

# 替换 var_names
adata.var_names = [mapping.get(sym, sym) for sym in symbols]

# 保存新文件
adata.write_h5ad("/data2/xiaoxinyu/project/embedding-cosine/single-spot/sc2_ensembl.h5ad")
print("✅ 已完成 gene symbol → Ensembl ID 转换并保存。")


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


36 input query terms found dup hits:	[('ARHGAP26', 2), ('BEND2', 2), ('CDR1', 2), ('CFHR3', 2), ('CXCL11', 2), ('DDIT3', 2), ('EPPK1', 2)
1886 input query terms found no hit:	['AARS', 'ABCA10', 'ABCA8', 'ABCB1', 'ABCC11', 'AC007906.2', 'AC010255.3', 'AC010325.1', 'AC011005.1


✅ 已完成 gene symbol → Ensembl ID 转换并保存。


## 添加region_label niche_label注释信息

In [ ]:
import anndata
import pandas as pd

# 读取 AnnData
ad = anndata.read_h5ad("/data2/xiaoxinyu/project/embedding-cosine/DLPFC/top2000/151507/stdata.h5ad")

# 读取 region_label 注释文件（设定分隔符为 tab，且没有表头）
df_region = pd.read_csv("/data1/xiaoxinyu/benchmark/DLPFC/annotation/151507_truth.txt",
                 sep="\t", header=None, names=["barcode", "region_label"])

# 读取 niche_label 注释文件（设定分隔符为 tab，且没有表头）
df_niche = pd.read_csv("/data1/xiaoxinyu/benchmark/DLPFC/annotation/barcode_layer_annotation.txt",
                 sep="\t", header=None, names=["barcode", "niche_label"])

# 设置索引为 barcode，以便与 AnnData.obs 对齐
df_region.set_index("barcode", inplace=True)
df_niche.set_index("barcode", inplace=True)

# 确保只选取在 AnnData 中存在的条目
df_region_filtered = df_region.loc[df_region.index.intersection(ad.obs_names)]
df_niche_filtered = df_niche.loc[df_niche.index.intersection(ad.obs_names)]

# 添加到 AnnData.obs 中
ad.obs["region_label"] = df_region_filtered["region_label"]
ad.obs["niche_label"] = df_niche_filtered["niche_label"]

# 检查缺失情况
missing_region = ad.obs["region_label"].isna().sum()
print(f"添加完成，region_label 缺失注释的条目数: {missing_region}")
missing_niche = ad.obs["niche_label"].isna().sum()
print(f"添加完成，niche_label 缺失注释的条目数: {missing_niche}")

# 如需保存新的 AnnData 文件
ad.write_h5ad("/data2/xiaoxinyu/project/embedding-cosine/DLPFC/top2000/151507/stdata_with_label.h5ad")


## 数据合并

In [20]:
import scanpy as sc

# 读取多个h5ad文件
file_paths = [
    '/data2/xiaoxinyu/project/embedding-cosine/DLPFC/top2000/151508/stdata_with_label.h5ad',
    '/data2/xiaoxinyu/project/embedding-cosine/DLPFC/top2000/151509/stdata_with_label.h5ad',
    '/data2/xiaoxinyu/project/embedding-cosine/DLPFC/top2000/151510/stdata_with_label.h5ad',
    '/data2/xiaoxinyu/project/embedding-cosine/DLPFC/top2000/151669/stdata_with_label.h5ad',
    '/data2/xiaoxinyu/project/embedding-cosine/DLPFC/top2000/151670/stdata_with_label.h5ad',
    '/data2/xiaoxinyu/project/embedding-cosine/DLPFC/top2000/151671/stdata_with_label.h5ad',
    '/data2/xiaoxinyu/project/embedding-cosine/DLPFC/top2000/151672/stdata_with_label.h5ad',
    '/data2/xiaoxinyu/project/embedding-cosine/DLPFC/top2000/151673/stdata_with_label.h5ad',
    '/data2/xiaoxinyu/project/embedding-cosine/DLPFC/top2000/151675/stdata_with_label.h5ad',
    '/data2/xiaoxinyu/project/embedding-cosine/DLPFC/top2000/151676/stdata_with_label.h5ad'
]

# 读取并合并所有文件
merged_adata = sc.read_h5ad(file_paths[0])
for file_path in file_paths[1:]:
    adata = sc.read_h5ad(file_path)
    merged_adata = merged_adata.concatenate(adata)

# 查看合并后的数据
print(merged_adata)

/tmp/ipykernel_2522913/231523622.py:21: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  merged_adata = merged_adata.concatenate(adata)
/tmp/ipykernel_2522913/231523622.py:21: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  merged_adata = merged_adata.concatenate(adata)
/tmp/ipykernel_2522913/231523622.py:21: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  merged_adata = merged_adata.concatenate(adata)
/tmp/ipykernel_2522913/231523622.py:21: FutureWarnin

AnnData object with n_obs × n_vars = 39782 × 17439
    obs: 'in_tissue', 'array_row', 'array_col', 'region_label', 'niche_label', 'batch'
    obsm: 'spatial'


In [21]:
save_path = '/data2/xiaoxinyu/project/embedding-cosine/DLPFC/top2000/merge_data/merged_stdata_with_label.h5ad'
merged_adata.write_h5ad(save_path)

In [1]:
import scanpy as sc
ad = sc.read_h5ad('/data2/xiaoxinyu/project/embedding-cosine/DLPFC/top2000/merge_data/merged_stdata_with_label.h5ad')

In [4]:
ad.obs_names[3]

'AAACAGAGCGACTCCT-1-0-0-0-0-0-0-0-0-0'

## 查看pt文件

In [2]:
import torch
ckpt = torch.load("/data2/xiaoxinyu/nicheformer/planB_stageA_ckpt/planB_stage1_ckpt.pt")
print("Checkpoint keys:", ckpt.keys())

Checkpoint keys: dict_keys(['heads_state_dict', 'backbone_state_dict', 'label_mapping', 'n_classes', 'hidden_dim'])


In [4]:
import torch

# data = torch.load("/data2/xiaoxinyu/project/gene_text_pairs/DLPFC/gene_text_pairs-mean.pt")
data = torch.load("/data2/xiaoxinyu/project/gene_text_pairs/DLPFC/gene_text_pairs_with_label.pt")
# 查看数据结构
print(f"数据类型: {type(data)}")
print(f"数据长度: {len(data)}")

# 查看第一个元素的结构
print("\n第一个元素的结构:")
print(data[0].keys())

数据类型: <class 'list'>
数据长度: 39782

第一个元素的结构:
dict_keys(['symbol', 'gene_embedding', 'text_embedding', 'region_label'])


In [5]:
# 查看具体内容示例
print("\n示例数据:")
print(f"基因符号: {data[0]['symbol']}")
print(f"基因嵌入维度: {len(data[0]['gene_embedding'])}")
print(f"文本嵌入维度: {len(data[0]['text_embedding'])}")
# print(f"Region 标签: {data[0]['region_label']}")


示例数据:
基因符号: TGAACTGCTATGACTT-1-1-0-0-0-0-0-0-0
基因嵌入维度: 1
文本嵌入维度: 3584


In [7]:
print(torch.tensor(data[0]["gene_embedding"]).shape)
print(torch.tensor(data[0]["text_embedding"]).shape)
# print(data["region_label"].dim())

torch.Size([1, 512])
torch.Size([3584])


In [8]:
print(data[0]["gene_embedding"])
print(data[0]["text_embedding"])

[[0.10056610405445099, -0.5766942501068115, 1.49311101436615, 0.14578638970851898, 0.5307971835136414, -0.7344838380813599, 1.0511170625686646, 0.6413503289222717, -0.5363997220993042, 0.5592153668403625, -1.005314826965332, -0.34156128764152527, 0.0759575143456459, 0.04627242684364319, 0.03834410384297371, -0.41831034421920776, -1.3197306394577026, -0.14005006849765778, 0.19991765916347504, 0.7730451226234436, -0.6399576663970947, -2.0874829292297363, -0.5943428874015808, -0.10013651847839355, 1.1202707290649414, -1.9152781963348389, 1.6756720542907715, -0.7717350721359253, -1.3720307350158691, 0.6170029640197754, 0.4669536352157593, 1.1887410879135132, -2.1334168910980225, -0.42079928517341614, -0.8884457349777222, -1.1668157577514648, -1.5649001598358154, 0.12509611248970032, 0.7077068090438843, 0.3834839165210724, -0.3138849139213562, 1.5449466705322266, 0.21370205283164978, -0.7707837820053101, 0.001641777460463345, 0.9984939098358154, -0.6823920607566833, 1.5428471565246582, 1.01

In [3]:
import torch
data_gene = torch.load("/data2/xiaoxinyu/project/gene_text_pairs/DLPFC/gene-embeddings/AAACAAGTATCTCCCA-1-1_gene.pt")
data_gene = data_gene.squeeze(0) 
print(data_gene.shape)
data_text = torch.load("/data2/xiaoxinyu/project/gene_text_pairs/DLPFC/text-embeddings/AAACAAGTATCTCCCA-1-1-0-0-0_text.pt")
print(data_text.shape)

torch.Size([512])
torch.Size([3584])
